In [ ]:
import json
import os
import sys

import pandas as pd
from scipy.sparse import load_npz

sys.path.append(os.path.relpath("src/food_analysis/core"))
import nlp

sys.path.append(os.path.relpath("src/food_analysis/utils"))
import tokenization

# From Scratch

In [ ]:
# Loading dataframe
data_recipe = pd.read_csv("data/raw/RAW_recipes.csv")

In [ ]:
# Extracting and saving tokens
data_text = tokenization.extract_text_from_df(data_recipe)
docs, stopwords = tokenization.extract_tokens_from_df(data_text)
data_recipe = tokenization.store_tokens_in_df(docs, stopwords, data_recipe)

output_tokens = "data/processed/data_recipe_with_tokens.csv"
data_recipe.to_csv(output_tokens, index=False, encoding="utf-8")

In [ ]:
# Processing the tokens to train the model

# Creating the bag of words
texts = [t for t in data_recipe["tokens"].tolist() if len(t) > 0]
vocabulary = nlp.create_vocabulary(texts)
recipe_counts = nlp.count_words(texts, vocabulary)

# Data saving
# save_npz("matrix.npz", recipe_counts)

# with open("dict.json", "w") as f:
#    json.dump(vocabulary, f)

# Processing search query

In [ ]:
# Processing the user search to get results

# Tokenization
query_text = ["Fajitas with guacamole"]  # String to be imported from the webapp
query_tokens = tokenization.extract_tokens_from_string(query_text)

# Vocabulary loading
with open("data/processed/dict.json", "r") as f:
    recipe_vocabulary = json.load(f)

# Bag of Words
query_counts = nlp.count_query(query_tokens, recipe_vocabulary)

# Computing results

In [ ]:
# Full BoW loading

bow_counts = load_npz("data/processed/matrix.npz")

# Doing TF-IDF
tf_idf_train, tf_idf_predict = nlp.tf_idf_search(bow_counts, query_counts)

# Getting results for the KNN
distances, indices = nlp.knn_search(tf_idf_train, 30, tf_idf_predict)
closest_recipes_df = nlp.send_recipes_id(data_recipe,indices)

closest_recipes_df.head(30)